# Slang Generation Demo

This demo will illustrate the basic workflow of the slang generation code accompanying the TACL paper *[A Computational Framework for Slang Generation](https://direct.mit.edu/tacl/article/doi/10.1162/tacl_a_00378/100687/A-Computational-Framework-for-Slang-Generation)* using slang definition data from Urban Dictionary (UD) and conventional definition data from WordNet.

To run this tutorial, you will need the following dependencies:

- Python 3
- Numpy
- Scipy
- tqdm
- NLTK
- Gensim
- PyTorch (torch)
- SBERT (sentence_transformers)
- [CatGO](https://github.com/zhewei-sun/CatGO)


In [1]:
import numpy as np
import torch
import shutil
import copy

We first create a symbolic link pointing to the library. You will need to change the destination if your code sits in a different directory. 

In [2]:
! ln -s ../Code slanggen

ln: failed to create symbolic link 'slanggen/Code': File exists


CatGO is a library that optimizes and runs models of categorization and can be obtained [here](https://github.com/zhewei-sun/CatGO). Once you have downloaded the code, please link it by replacing the target directory of the simlink below. 

In [3]:
! ln -s ../../CatGO CatGO
import nltk
nltk.download('stopwords')

ln: failed to create symbolic link 'CatGO/CatGO': File exists


[nltk_data] Downloading package stopwords to /home/kefan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
from slanggen.util import *
from slanggen.dataloader import WN_Dataset, Urban_Dataset, OSD_Dataset, ZH_Dataset
from slanggen.encoder import FTEncoder, FTCachedEncoder, SBertEncoder, SenseEncoder, dump_vanilla_embeddings
from slanggen.contrastive import SlangGenTrainer
from slanggen.model import SlangGenModel

/home/kefan/miniconda3/envs/slanggen/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Specify a PyTorch device if necessary:

In [5]:
torch.cuda.set_device(0)

Load conventional definition data using the builtin dataloaders. The *.npy* file loaded below contains a pre-processed version of WordNet definition sentences for all words that appear in both WordNet and UD.

In [6]:
# wn_data = WN_Dataset('mix_conv_data_ENZH.npy') 
# wn_data = WN_Dataset('mix_conv_data_ENRU.npy')
wn_data = WN_Dataset('mix_conv_data_all.npy')

# wn_data = WN_Dataset('mix_multilingual_conv_data_ENZH.npy') # mixed conv data
# wn_data = WN_Dataset('mix_multilingual_conv_data_ENRU.npy') # mixed conv data
# wn_data = WN_Dataset('mix_multilingual_conv_data_all.npy') # mixed conv data

print(wn_data)

# EN conv dataset, used for prediction test
en_conv_data = WN_Dataset('wordnet_urban.npy')
print(en_conv_data)

# ZH conv dataset, used for prediction test
zh_conv_data = WN_Dataset('ZH_conv_data.npy')
print(zh_conv_data)
ZH_zh_conv_data = WN_Dataset('ZH_zh_conv_data.npy')
print(ZH_zh_conv_data)

# RU conv dataset, used for prediction test
ru_conv_data = WN_Dataset('RU_conv_data.npy')
print(ru_conv_data)
RU_ru_conv_data = WN_Dataset('RU_ru_conv_data.npy')
print(RU_ru_conv_data)

Dataset Name: 
Total Definition Entries: 11363
Vocab Size: 3068

Dataset Name: 
Total Definition Entries: 9759
Vocab Size: 1464

Dataset Name: 
Total Definition Entries: 643
Vocab Size: 643

Dataset Name: 
Total Definition Entries: 643
Vocab Size: 643

Dataset Name: 
Total Definition Entries: 961
Vocab Size: 961

Dataset Name: 
Total Definition Entries: 961
Vocab Size: 961



Load slang definition data. The *.npy* file loaded below is a pre-processed version of the data released in this repository.

In [7]:
# We also want to try mixed data (EN + ZH) / (EN + RU)

# mix_data = OSD_Dataset('mix_slang_data_ENZH.npy', wn_data)
# mix_data = OSD_Dataset('mix_slang_data_ENRU.npy', wn_data)
mix_data = OSD_Dataset('mix_slang_data_all.npy', wn_data)

# mix_data = OSD_Dataset('mix_multilingual_slang_data_ENZH.npy', wn_data) # mixed slang data
# mix_data = OSD_Dataset('mix_multilingual_slang_data_ENRU.npy', wn_data) # mixed slang data
# mix_data = OSD_Dataset('mix_multilingual_slang_data_all.npy', wn_data) # mixed slang data

print(mix_data)

# EN/ZH slang dataset, used for prediction test
en_slang_data = OSD_Dataset('OSD_data.npy', en_conv_data)
zh_slang_data = ZH_Dataset('ZH_slang_data.npy', zh_conv_data)
ru_slang_data = ZH_Dataset('RU_slang_data.npy', ru_conv_data)

# Test the performance on other languages
zh_data = ZH_Dataset('ZH_zh_slang_data.npy', ZH_zh_conv_data)
print(zh_data)
ru_data = ZH_Dataset('RU_ru_slang_data.npy', RU_ru_conv_data)
print(ru_data)

Dataset Name: 
Total Definition Entries: 3453
Vocab Size: 2030

Dataset Name: 
Total Definition Entries: 643
Vocab Size: 643

Dataset Name: 
Total Definition Entries: 1016
Vocab Size: 961



If you wish to use your own dataset, please create a dataloader object inheriting either *ConvDataset* or *SlangDataset* abstract classes found in *dataloader.py* and following the example data specifications in *dataloader.WN_Dataset* and *dataloader.Urban_Dataset*.

Now let's create a directory to store our results and load in some pre-generated data indices for train-test split:

In [8]:
shutil.rmtree('Results')  
create_directory('Results')

In [9]:
out_dir='Results/'

dataset_mix = mix_data
# slang_inds_mix = DataIndex(np.load('train_ind_mix_ENZH.npy'), np.load('dev_ind_mix_ENZH.npy'), np.load('test_ind_mix_ENZH.npy'))
# slang_inds_mix = DataIndex(np.load('train_ind_mix_ENRU.npy'), np.load('dev_ind_mix_ENRU.npy'), np.load('test_ind_mix_ENRU.npy'))
slang_inds_mix = DataIndex(np.load('train_ind_mix_all.npy'), np.load('dev_ind_mix_all.npy'), np.load('test_ind_mix_all.npy'))

# slang_inds_mix = DataIndex(np.load('train_ind_mix_multilingual_ENZH.npy'), np.load('dev_ind_mix_multilingual_ENZH.npy'), np.load('test_ind_mix_multilingual_ENZH.npy'))
# slang_inds_mix = DataIndex(np.load('train_ind_mix_multilingual_ENRU.npy'), np.load('dev_ind_mix_multilingual_ENRU.npy'), np.load('test_ind_mix_multilingual_ENRU.npy'))
# slang_inds_mix = DataIndex(np.load('train_ind_mix_multilingual_all.npy'), np.load('dev_ind_mix_multilingual_all.npy'), np.load('test_ind_mix_multilingual_all.npy'))

# For prediction tests
dataset_mix_en = en_slang_data
slang_inds_en = DataIndex(np.load('train_ind_osd.npy'), np.load('dev_ind_osd.npy'), np.load('test_ind_osd.npy'))

dataset_mix_zh = zh_slang_data
slang_inds_zh = DataIndex(np.load('train_ind_zh.npy'), np.load('dev_ind_zh.npy'), np.load('test_ind_zh.npy'))

dataset_mix_ru = ru_slang_data
slang_inds_ru = DataIndex(np.load('train_ind_ru.npy'), np.load('dev_ind_ru.npy'), np.load('test_ind_ru.npy'))

#Test the performance on other languages
dataset_zh_zh = zh_data
slang_inds_zh_zh = DataIndex(np.load('train_ind_zh_zh.npy'), np.load('dev_ind_zh_zh.npy'), np.load('test_ind_zh_zh.npy'))

dataset_ru_ru = ru_data
slang_inds_ru_ru = DataIndex(np.load('train_ind_ru_ru.npy'), np.load('dev_ind_ru_ru.npy'), np.load('test_ind_ru_ru.npy'))

The following encoder objects initializes a fastText encoder used for collaborative filtering. *FTEncoder* can be used to read in the original fastText embedding file. For efficiency, we have cached the words we need and use a cached encoder instead.

ft_encoder = FTEncoder('path to crawl-300d-2M-subword.vec')

In [10]:
ft_encoder = FTCachedEncoder('ft_embed_cache_Urban.pickle')

The following commands sets up the contrastive trainer and the slang generation model:

In [11]:
trainer_mix = SlangGenTrainer(dataset_mix, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)

You can modify the 'embed_name' param below to choose the sense encoding model, here is a list of supported models:
- bert-base-nli-mean-tokens -> 'SBERT_contrastive' (default)
- sentence-t5-base -> 'SBERT_t5'
- paraphrase-multilingual-MiniLM-L12-v2 -> 'SBERT-multilingual-MiniLM-L12-v2'
- LaBSE -> 'SBERT_LaBSE'
- paraphrase-multilingual-mpnet-base-v2 -> 'SBERT_mpnet'
- intfloat/multilingual-e5-base -> 'SBERT_e5_base'
- intfloat/multilingual-e5-large -> 'SBERT_e5_large'

In [12]:
model_mix = SlangGenModel(trainer_mix, data_dir=out_dir, embed_name='SBERT_mpnet')

params = {'embed_name':'SBERT_mpnet', 'out_name':'predictions', 'model':'prototype', 'prior':None, 'prior_name':'uniform', 'contr_params':None}

Invoke *model.train_contrastive* to train the contrastively learned sense embedding model:

Note: you can set mode to 'head' to train only the triplet head, or 'whole' to train both the sense encoding and the head. Also it's optional to change the fold_name if using a new dataset.

In [13]:
model_mix.train_contrastive(slang_inds_mix, fold_name='mix_wn', params=params, mode='head')

Generating contrative pairs...


100%|███████████████████████████████████████████████████████████████████████| 172/172 [00:11<00:00, 15.38it/s]


Complete!
Training contrastive model with head
Generating triplet data for contrastive training...
Sampled 108732 Triplets
Sampled 6254 Triplets
Complete!


/home/kefan/miniconda3/envs/slanggen/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Epoch 1/4 - Training: 100%|█████████████████████████████████████████████| 13587/13587 [03:25<00:00, 66.10it/s]


Epoch 1 average training loss: 0.4947


...

Epoch 4 average training loss: 0.2860


Epoch 4/4 - Validation: 100%|███████████████████████████████████████████████| 782/782 [00:11<00:00, 70.29it/s]


Epoch 4 average validation loss: 0.3981
New best model saved with validation loss: 0.3981
Training completed. Best validation loss: 0.3981
Cleared GPU cache before loading model


/home/kefan/miniconda3/envs/slanggen/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Encoding sense definitions...
Complete!


In [14]:
# Now we want to see how the mixed dataset performs on each language

# Step 1: Create symbolic links
!mkdir -p Results/mix_wn_en/SBERT_data
!mkdir -p Results/mix_wn_zh/SBERT_data
!mkdir -p Results/mix_wn_ru/SBERT_data

!cp Results/mix_wn/SBERT_data/SBERT_mpnet_with_head.pt Results/mix_wn_en/SBERT_data/
!cp Results/mix_wn/SBERT_data/SBERT_mpnet_with_head.pt Results/mix_wn_zh/SBERT_data/
!cp Results/mix_wn/SBERT_data/SBERT_mpnet_with_head.pt Results/mix_wn_ru/SBERT_data/

# !cp Results/mix_wn/SBERT_data/SBERT_contrastive_with_head.pt Results/mix_wn_en/SBERT_data/
# !cp Results/mix_wn/SBERT_data/SBERT_contrastive_with_head.pt Results/mix_wn_zh/SBERT_data/

# !cp Results/mix_wn/SBERT_data/SBERT_contrastive_whole_finetuned.pt Results/mix_wn_en/SBERT_data/
# !cp Results/mix_wn/SBERT_data/SBERT_contrastive_whole_finetuned.pt Results/mix_wn_zh/SBERT_data/

params['embed_name'] = 'SBERT_mpnet' 

trainer_mix_en = SlangGenTrainer(dataset_mix_en, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)
model_mix_en   = SlangGenModel(trainer_mix_en, data_dir=out_dir, embed_name='SBERT_mpnet')

trainer_mix_zh = SlangGenTrainer(dataset_mix_zh, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)
model_mix_zh   = SlangGenModel(trainer_mix_zh, data_dir=out_dir, embed_name='SBERT_mpnet')

trainer_mix_ru = SlangGenTrainer(dataset_mix_ru, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)
model_mix_ru   = SlangGenModel(trainer_mix_ru, data_dir=out_dir, embed_name='SBERT_mpnet')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

In [15]:
trainer_mix_en.get_trained_embeddings(slang_inds_en, fold_name='mix_wn_en', model_path='SBERT_mpnet')
trainer_mix_zh.get_trained_embeddings(slang_inds_zh, fold_name='mix_wn_zh', model_path='SBERT_mpnet')
trainer_mix_ru.get_trained_embeddings(slang_inds_ru, fold_name='mix_wn_ru', model_path='SBERT_mpnet')

model_mix_en.train_categorization(slang_inds_en, fold_name='mix_wn_en', params=params)
results_mix_en = model_mix_en.get_results(fold_name='mix_wn_en', mode='train', params=params)

params_zh = {'embed_name':'SBERT_mpnet', 'out_name':'predictions', 'model':'prototype', 'prior':None, 'prior_name':'uniform', 'contr_params':None}
params_ru = {'embed_name':'SBERT_mpnet', 'out_name':'predictions', 'model':'prototype', 'prior':None, 'prior_name':'uniform', 'contr_params':None}

model_mix_zh.train_categorization(slang_inds_zh, fold_name='mix_wn_zh', params=params_zh)
results_mix_zh = model_mix_zh.get_results(fold_name='mix_wn_zh', mode='train', params=params_zh)

model_mix_ru.train_categorization(slang_inds_ru, fold_name='mix_wn_ru', params=params_ru)
results_mix_ru = model_mix_ru.get_results(fold_name='mix_wn_ru', mode='train', params=params_ru)

Cleared GPU cache before loading model
Encoding sense definitions...
Complete!
Cleared GPU cache before loading model
Encoding sense definitions...
Complete!
Cleared GPU cache before loading model
Encoding sense definitions...
Complete!
Pre-processing Distances...
Pre-processing Complete!
Optimizing Kernels...


100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  7.59it/s]


Pre-processing Distances...
Pre-processing Complete!
Optimizing Kernels...


100%|██████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 172.21it/s]


Pre-processing Distances...
Pre-processing Complete!
Optimizing Kernels...


100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 23.56it/s]


In [16]:
print('Performance on train split - EN')
N_train_dev_mix_en = dataset_mix_en.N_total - slang_inds_en.test.shape[0]
train_rankings_mix_en = get_rankings(results_mix_en, np.arange(N_train_dev_mix_en), dataset_mix_en.vocab_ids[np.concatenate(slang_inds_en)])
np.mean(get_roc(train_rankings_mix_en, dataset_mix_en.V))

Performance on train split - EN


0.7889082958764901

In [17]:
print('Performance on train split - ZH')
N_train_dev_mix_zh = dataset_mix_zh.N_total - slang_inds_zh.test.shape[0]
train_rankings_mix_zh = get_rankings(results_mix_zh, np.arange(N_train_dev_mix_zh), dataset_mix_zh.vocab_ids[np.concatenate(slang_inds_zh)])
np.mean(get_roc(train_rankings_mix_zh, dataset_mix_zh.V))

Performance on train split - ZH


0.947025921507259

In [18]:
print('Performance on train split - RU')
N_train_dev_mix_ru = dataset_mix_ru.N_total - slang_inds_ru.test.shape[0]
train_rankings_mix_ru = get_rankings(results_mix_ru, np.arange(N_train_dev_mix_ru), dataset_mix_ru.vocab_ids[np.concatenate(slang_inds_ru)])
np.mean(get_roc(train_rankings_mix_ru, dataset_mix_ru.V))

Performance on train split - RU


0.7744875067470313

In [19]:
model_mix_en.predict_testset(slang_inds_en, fold_name='mix_wn_en', params=params)
results_mix_en = model_mix_en.get_results(fold_name='mix_wn_en', mode='test', params=params)

model_mix_zh.predict_testset(slang_inds_zh, fold_name='mix_wn_zh', params=params_zh)
results_mix_zh = model_mix_zh.get_results(fold_name='mix_wn_zh', mode='test', params=params_zh)

model_mix_ru.predict_testset(slang_inds_ru, fold_name='mix_wn_ru', params=params_ru)
results_mix_ru = model_mix_ru.get_results(fold_name='mix_wn_ru', mode='test', params=params_ru)

Params: {'embed_name': 'SBERT_mpnet', 'out_name': 'predictions', 'model': 'prototype', 'prior': None, 'prior_name': 'uniform', 'contr_params': None}
Pre-processing Distances...
Pre-processing Complete!
Optimizing Kernels...


100%|██████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 342.95it/s]


Params: {'embed_name': 'SBERT_mpnet', 'out_name': 'predictions', 'model': 'prototype', 'prior': None, 'prior_name': 'uniform', 'contr_params': None}
Pre-processing Distances...
Pre-processing Complete!
Optimizing Kernels...


100%|█████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 2120.48it/s]


Params: {'embed_name': 'SBERT_mpnet', 'out_name': 'predictions', 'model': 'prototype', 'prior': None, 'prior_name': 'uniform', 'contr_params': None}
Pre-processing Distances...
Pre-processing Complete!
Optimizing Kernels...


100%|██████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 689.17it/s]


In [20]:
print('Performance on test split - EN')
inds = slang_inds_en.test                      
labels = dataset_mix_en.vocab_ids                 
test_rankings_mix_en = get_rankings(results_mix_en, inds, labels)
np.mean(get_roc(test_rankings_mix_en, dataset_mix_en.V))

Performance on test split - EN


0.7530592469545958

In [21]:
print('Performance on test split - ZH')
inds = slang_inds_zh.test                      
labels = dataset_mix_zh.vocab_ids                 
test_rankings_mix_zh = get_rankings(results_mix_zh, inds, labels)
np.mean(get_roc(test_rankings_mix_zh, dataset_mix_zh.V))

Performance on test split - ZH


0.937452774746033

In [22]:
print('Performance on test split - RU')
inds = slang_inds_ru.test                      
labels = dataset_mix_ru.vocab_ids                 
test_rankings_mix_ru = get_rankings(results_mix_ru, inds, labels)
np.mean(get_roc(test_rankings_mix_ru, dataset_mix_ru.V))

Performance on test split - RU


0.7836471956740443

In [23]:
# Test how the model perform on different datasets in their own languages
params['embed_name'] = 'SBERT_mpnet' 

!mkdir -p Results/test_zh/SBERT_data
!mkdir -p Results/test_ru/SBERT_data

!cp Results/mix_wn/SBERT_data/SBERT_mpnet_with_head.pt Results/test_zh/SBERT_data/
# !cp Results/osd_wn/SBERT_data/SBERT_mpnet_whole_finetuned.pt Results/test_zh/SBERT_data/

trainer_test_zh = SlangGenTrainer(zh_data, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)
model_test_zh   = SlangGenModel(trainer_test_zh, data_dir=out_dir, embed_name='SBERT_mpnet')

trainer_test_zh.get_trained_embeddings(slang_inds_zh_zh, fold_name='test_zh', model_path='SBERT_mpnet')

model_test_zh.train_categorization(slang_inds_zh_zh, fold_name='test_zh', params=params)
results_zh = model_test_zh.get_results(fold_name='test_zh', mode='train', params=params)

print('Performance on train split - ZH')
N_train_dev_zh = zh_data.N_total - slang_inds_zh_zh.test.shape[0]
train_rankings_zh = get_rankings(results_zh, np.arange(N_train_dev_zh), zh_data.vocab_ids[np.concatenate(slang_inds_zh_zh)])
np.mean(get_roc(train_rankings_zh, zh_data.V))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Cleared GPU cache before loading model
Encoding sense definitions...
Complete!
Pre-processing Distances...
Pre-processing Complete!
Optimizing Kernels...


100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 55.49it/s]

Performance on train split - ZH


0.9626621684019244

In [24]:
model_test_zh.predict_testset(slang_inds_zh_zh, fold_name='test_zh', params=params)
results_zh = model_test_zh.get_results(fold_name='test_zh', mode='test', params=params)

print('Performance on test split - ZH')
inds = slang_inds_zh_zh.test                      
labels = zh_data.vocab_ids                 
test_rankings_zh = get_rankings(results_zh, inds, labels)
np.mean(get_roc(test_rankings_zh, zh_data.V))

Params: {'embed_name': 'SBERT_mpnet', 'out_name': 'predictions', 'model': 'prototype', 'prior': None, 'prior_name': 'uniform', 'contr_params': None}
Pre-processing Distances...
Pre-processing Complete!
Optimizing Kernels...


100%|█████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1396.24it/s]

Performance on test split - ZH


0.9765010351966874

In [25]:
!cp Results/mix_wn/SBERT_data/SBERT_mpnet_with_head.pt Results/test_ru/SBERT_data/
# !cp Results/osd_wn/SBERT_data/SBERT_mpnet_whole_finetuned.pt Results/test_ru/SBERT_data/

trainer_test_ru = SlangGenTrainer(ru_data, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)
model_test_ru   = SlangGenModel(trainer_test_ru, data_dir=out_dir, embed_name='SBERT_mpnet')

trainer_test_ru.get_trained_embeddings(slang_inds_ru_ru, fold_name='test_ru', model_path='SBERT_mpnet')

model_test_ru.train_categorization(slang_inds_ru_ru, fold_name='test_ru', params=params)
results_ru = model_test_ru.get_results(fold_name='test_ru', mode='train', params=params)

print('Performance on train split - RU')
N_train_dev_ru = ru_data.N_total - slang_inds_ru_ru.test.shape[0]
train_rankings_ru = get_rankings(results_ru, np.arange(N_train_dev_ru), ru_data.vocab_ids[np.concatenate(slang_inds_ru_ru)])
np.mean(get_roc(train_rankings_ru, ru_data.V))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Cleared GPU cache before loading model
Encoding sense definitions...
Complete!
Pre-processing Distances...
Pre-processing Complete!
Optimizing Kernels...


100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 22.18it/s]

Performance on train split - RU


0.6990998719623256

In [26]:
model_test_ru.predict_testset(slang_inds_ru_ru, fold_name='test_ru', params=params)
results_ru = model_test_ru.get_results(fold_name='test_ru', mode='test', params=params)

print('Performance on test split - RU')
inds = slang_inds_ru_ru.test                      
labels = ru_data.vocab_ids                 
test_rankings_ru = get_rankings(results_ru, inds, labels)
np.mean(get_roc(test_rankings_ru, ru_data.V))

Params: {'embed_name': 'SBERT_mpnet', 'out_name': 'predictions', 'model': 'prototype', 'prior': None, 'prior_name': 'uniform', 'contr_params': None}
Pre-processing Distances...
Pre-processing Complete!
Optimizing Kernels...


100%|██████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 644.09it/s]

Performance on test split - RU


0.7199089891397583